In [5]:
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.conversation import conv_templates, SeparatorStyle
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import tokenizer_image_token, get_model_name_from_path, KeywordsStoppingCriteria, process_images

In [6]:
import argparse
import torch
import os
import json
from tqdm import tqdm
import shortuuid
from PIL import Image
import math
from transformers import set_seed, logging

In [4]:
from llava.model.builder import load_pretrained_model
tokenizer, model, image_processor, context_len = load_pretrained_model(
        model_path='/work/vlmwork/LLaVA-Med/llavamodel',
        model_base=None,
        model_name='llava-med-v1.5-mistral-7b'
 )


/root/miniconda/envs/llava-med/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 4/4 [00:13<00:00,  3.37s/it]
Some weights of the model checkpoint at /work/vlmwork/LLaVA-Med/llavamodel were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.encoder.layers.5.self_attn.q_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.13.self_attn.out_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.22.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.9.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.5.self_attn.v_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.l

In [102]:
MANIFEST_PATH = 'data/vlmdata/manifest.jsonl'
IMAGE_FOLDER = os.path.join(f'data/vlmdata')
OUTPUT_PATH = f'outputs'
PROMPTS = [
    "Describe this ophthalmic image briefly. Answer in json format like this: {'Description': 'your description', 'Diagnosis': 'diagnosis label'}",
    "What abnormalities are visible? If none are visible, say so clearly. Answer in json format like this: {'Description': 'your description', 'Diagnosis': 'diagnosis label'}",
    "Give the most likely diagnosis and a one-sentence justification based only on visible evidence. Answer in json format like this: {'Description': 'your description', 'Diagnosis': 'diagnosis label'}"
]

In [103]:
def load_image(path):
    return Image.open(path)


In [104]:
data = []
with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))



In [105]:
results = []

for sample in data:
    img = load_image(sample['image'])
    image_tensor = process_images([img], image_processor, model.config)[0]

    
    for prompt in PROMPTS:
        prompt = prompt.replace(DEFAULT_IMAGE_TOKEN, '').strip()
        if model.config.mm_use_im_start_end:
            prompt = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + prompt
        else:
            prompt = DEFAULT_IMAGE_TOKEN + '\n' + prompt
        
        conv = conv_templates['mistral_instruct'].copy()
        conv.append_message(conv.roles[0], prompt)
        conv.append_message(conv.roles[1], None)
        prompt = conv.get_prompt()    

        input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()

        stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
        keywords = [stop_str]
        stopping_criteria = KeywordsStoppingCriteria(keywords, tokenizer, input_ids)


        with torch.inference_mode():
            output_ids = model.generate(
                input_ids,
                images=image_tensor.unsqueeze(0).half().cuda(),
                do_sample=True,
                temperature=0.2,
                top_p=None,
                num_beams=1,
                # no_repeat_ngram_size=3,
                max_new_tokens=1024,
                use_cache=True
            )

        outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

        results.append({
            "image": sample["image"],
            "label": sample["label"],
            "prompt": prompt,
            "response": outputs
        })

    

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attentio

In [106]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("outputs/llava-med-results.csv", index=False)

df.head()

,image,label,prompt,response
0,data/vlmdata/img_0.png,no_diabetic_retinopathy,[INST] <image>\nDescribe this ophthalmic image...,"The image is an ophthalmic image, which means ..."
1,data/vlmdata/img_0.png,no_diabetic_retinopathy,[INST] <image>\nWhat abnormalities are visible...,The image shows no abnormalities.
2,data/vlmdata/img_0.png,no_diabetic_retinopathy,[INST] <image>\nGive the most likely diagnosis...,{'Description': 'The image shows a white-cente...
3,data/vlmdata/img_1.png,no_diabetic_retinopathy,[INST] <image>\nDescribe this ophthalmic image...,{'Description': 'The ophthalmic image shows a ...
4,data/vlmdata/img_1.png,no_diabetic_retinopathy,[INST] <image>\nWhat abnormalities are visible...,The image shows no abnormalities.


In [ ]:
qs = "Describe this ophthalmic image briefly."
qs = qs.replace(DEFAULT_IMAGE_TOKEN, '').strip()

if model.config.mm_use_im_start_end:
    qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + qs
else:
    qs = DEFAULT_IMAGE_TOKEN + '\n' + qs

conv = conv_templates['mistral_instruct'].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()    

input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()

image_file = 'img_0.png'
image = Image.open(os.path.join(IMAGE_FOLDER, image_file))

image_tensor = process_images([image], image_processor, model.config)[0]

stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
keywords = [stop_str]
stopping_criteria = KeywordsStoppingCriteria(keywords, tokenizer, input_ids)



In [71]:
prompt

'[INST] <image>\nDescribe this ophthalmic image briefly. [/INST]'

In [72]:
with torch.inference_mode():
    output_ids = model.generate(
        input_ids,
        images=image_tensor.unsqueeze(0).half().cuda(),
        do_sample=True,
        temperature=0.2,
        top_p=None,
        num_beams=1,
        # no_repeat_ngram_size=3,
        max_new_tokens=1024,
        use_cache=True
    )

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [73]:
outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

In [74]:
outputs

"This ophthalmic image is a fundus photograph, which is a photograph of the retina and surrounding structures in the eye. The image is taken using a fundus camera, which is a specialized camera designed for this purpose. The fundus photograph provides a detailed view of the eye's internal structures, allowing healthcare professionals to examine the retina, optic nerve, and other important structures for any abnormalities or signs of disease."